# Hex-grain ignition likelihood: what beats chance, and what does not

**The W6 product.** *Where in my region are fires most likely to start?* This notebook builds the
target, then asks the question that had not been asked anywhere in the project until now: **how
much of our apparent predictive skill is real, measured against a naive baseline?**

Every earlier comparison in this project scored a model against *persistence* or against a *global
rate*. Both are already-informed baselines. Neither answers "is this better than chance," and
without that number a reader cannot tell whether the whole exercise has produced anything.

**Why starts, and why this needs no perimeters.** [`10_hex_burn_demo.ipynb`](10_hex_burn_demo.ipynb)
established the point-vs-area confound: FPA-FOD stores a *pinpoint* ignition location but
`FIRE_SIZE` describes an *area*, which is what forced the MTBS perimeter build. Read in the other
direction, that same asymmetry makes an **ignition-count** target cheap — an ignition location is
exactly what the record stores correctly. So this target uses raw points, all 2.27M fires, and no
MTBS join at all. Distributing an ignition across a perimeter would smear one start across ~26
hexes and count it 26 times.

**Human and Natural are modelled separately** (student decision, W6). They are different processes:
human starts track roads and settlement — largely *static* geography — while natural starts track
lightning and fuel dryness. Pooling would let the 78% human mass dilute the climate signal.

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, "../src")
import hex_panel as hp
from config import ProjectConfig

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data

# Seeded so the shuffled-persistence control below is reproducible.
RNG = np.random.default_rng(0)

# The trailing window for the starts baseline lives in src/hex_panel.py, not
# here. Every earlier version of this analysis re-typed the persistence idiom
# inline, and the ablation numbers moved between runs as a result — see
# "Why the baseline lives in a module" below.
print(f"forward-chaining split: train < {cfg.test_start}, test >= {cfg.test_start}")
print(f"persistence window: k={hp.STARTS_K}")

## The target

`src/hex_ignitions.py` counts ignitions per `(hex_id, season_idx)`, one column per cause surface.

Two design points are load-bearing:

- **The panel is densified.** Most hex-seasons have no ignition, and "no fire started here" is an
  observation, not a missing value. Building the target from the fire table alone would silently
  condition the entire model on already-burning cells.
- **`log_area` is an exposure offset, not a feature.** A raw count target would substantially
  rediscover which hexes are large. This distinction is not cosmetic — see the offset section
  below, where getting it wrong inflated every prediction by 68%.

In [ ]:
panel = pd.read_parquet(DATA / "hex_ignitions.parquet")

panel["season_year"] = 1992 + (panel["season_idx"] // 4)
panel["season_ord"] = panel["season_idx"] % 4
panel = panel.sort_values(["hex_id", "season_idx"]).reset_index(drop=True)

print(f"panel: {len(panel):,} hex-season cells "
      f"({panel['hex_id'].nunique():,} hexes x {panel['season_idx'].nunique()} seasons)")
print(f"season_year range: {panel['season_year'].min()}-{panel['season_year'].max()}\n")

for c in ["natural", "human", "unknown", "total"]:
    col = f"starts_{c}"
    print(f"  starts_{c:<8} {panel[col].sum():>10,.0f} ignitions | "
          f"{(panel[col] > 0).mean():6.2%} of cells nonzero")

## The count is overdispersed and zero-inflated

Before choosing a loss, look at the distribution. A Poisson likelihood assumes variance equals the
mean; both branches violate that badly, and both have far more zeros than Poisson allows.

This is recorded here because it explains a modelling failure later in the notebook: Poisson
deviance as a *scoring metric* penalises each surprising nonzero far more than an overdispersed
process warrants.

In [ ]:
rows = []
for t in ["natural", "human"]:
    y = panel[f"starts_{t}"]
    rows.append({
        "branch": t,
        "mean": y.mean(),
        "variance": y.var(),
        "var/mean": y.var() / y.mean(),
        "actual P(0)": (y == 0).mean(),
        "Poisson P(0)": np.exp(-y.mean()),
        "max": int(y.max()),
    })
disp = pd.DataFrame(rows).set_index("branch")
print(disp.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nvar/mean of 1.0 would be Poisson. Human is 9.3x overdispersed, and Poisson")
print("expects 65.7% zeros where the data has 85.3% — a 20-point miss.")

## The persistence baseline — and why it lives in a module

The project's method commitment: *"region-season = its own last occurrence."* At this grain a hex's
next-season ignition count is the trailing mean of its own **same-season** history — same hex, same
season-of-year, strictly earlier seasons only.

**This baseline is not computed in this notebook.** It is computed once in
[`../src/hex_panel.py`](../src/hex_panel.py), which delegates to
[`../src/trailing.py`](../src/trailing.py) — the W5 module that asserts the sort invariant and
returns index-aligned output.

That indirection is the fix for a real failure. Earlier versions of this analysis re-typed the
`groupby → shift(1) → rolling(k).mean()` idiom inline on every run, roughly eight times across
throwaway scripts. The ablation numbers moved between runs — the same burn-history rung scored
−30%, −115% and +3.4% — and the cause was not the data but the harness being silently re-specified
each time. One definition, called everywhere, is what makes the ladder below reproducible.

The panel is **cached** (`hex_panel_modelling.parquet`), so the baseline is quoted rather than
recomputed.

In [ ]:
panel = hp.build_cached(DATA)
panel = hp.add_climate_anomalies(panel, cfg=cfg)

train, test = hp.split(panel, cfg=cfg)
print(f"panel {panel.shape} | train {train.sum():,} | test {test.sum():,}\n")

print("Cached persistence floors (Spearman, held-out):")
print(hp.baseline_scores(DATA).to_string(index=False))

## The question that had not been asked: is any of this better than chance?

Four baselines, scored on the held-out tail. The third is the one that matters.

| baseline | what it holds fixed | what it destroys |
| --- | --- | --- |
| uniform (train mean) | the overall rate | all spatial structure |
| random noise | nothing | everything |
| **shuffled persistence** | **the exact set of predicted values** | **only the hex-to-hex mapping** |
| persistence k=7 | — | — |

**Shuffled persistence is the honest null.** It carries the identical distribution of predictions
and breaks only *which hex gets which number*. If real persistence beats it, the skill is genuinely
spatial rather than an artifact of predicting plausible-looking magnitudes.

Two metrics, deliberately: **MAE** (magnitude accuracy) and **Spearman** (rank accuracy). They
disagree, and the disagreement matters — for a *siting* product the planner needs to know which
hexes to rank first, so Spearman is the metric that matches the decision. Neither assumes a
likelihood, which keeps this section independent of the Poisson problem above.

In [ ]:
def mae(y, p):
    return float(np.mean(np.abs(y - p)))


def rank_corr(y, p):
    """Spearman: does the prediction order the hexes correctly?"""
    return float(spearmanr(y, p).statistic)


results = []
for t in ["natural", "human"]:
    y_all = panel[f"starts_{t}"].to_numpy(float)
    pers_all = panel[f"pers_{t}"].to_numpy()

    # Score only where persistence is defined, so every baseline is scored on an
    # identical cell population — otherwise the comparison is not like-for-like.
    ok = test & np.isfinite(pers_all)
    y, pers = y_all[ok], pers_all[ok]

    baselines = {
        "uniform (train mean)": np.full(y.shape, y_all[train].mean()),
        "random noise": RNG.random(y.shape),
        "SHUFFLED persistence": RNG.permutation(pers),
        f"persistence k={hp.STARTS_K}": pers,
    }
    for label, pred in baselines.items():
        results.append({
            "branch": t, "baseline": label, "n_test": int(ok.sum()),
            "MAE": mae(y, pred), "Spearman": rank_corr(y, pred),
        })

scores = pd.DataFrame(results)
print("Held-out season_year >= 2010. Neither metric assumes a likelihood.\n")
print(scores.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

**Finding — the skill is real, and it is spatial.**

Shuffled persistence holds the same predicted values and destroys only the hex-to-hex mapping.
Spearman collapses from **+0.34 to −0.001** (Natural) and **+0.53 to −0.001** (Human). The
information lives in *which hex gets which number*, which is exactly what a siting product needs.

**Human ranks better than Natural** (+0.53 vs +0.34). That is the expected direction rather than a
surprise: human ignition tracks roads and settlement, which are near-static across 29 years, so a
hex's own history is close to a complete description. Natural ignition depends on where lightning
happens to strike in a given season, which history cannot see.

Note that MAE and Spearman rank the baselines differently — uniform beats random noise on MAE while
being useless for ranking. Reporting only MAE would have hidden the entire result.

## The exposure offset is not optional

`log_area` must enter as an **offset** (modelling starts *per unit area*), not as an ordinary
predictor. This cell demonstrates the difference, because getting it wrong is silent: the model
still trains, still predicts non-negative counts, and still looks reasonable.

Passing `log_area` as a plain feature lets the tree treat hex size as a free parameter to fit,
rather than as known exposure to divide by.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor


def poisson_dev(y, mu):
    mu = np.clip(mu, 1e-6, None)
    return float(2 * np.mean(np.where(y > 0, y * np.log(y / mu), 0.0) - (y - mu)))


y_all = panel["starts_natural"].to_numpy(float)
pers_all = panel["pers_natural"].to_numpy()
off_all = panel["log_area"].to_numpy()
ok = test & np.isfinite(pers_all)

demo = []
for label, X in [
    ("persistence only", pers_all.reshape(-1, 1)),
    ("+ log_area as a FEATURE (wrong)", np.column_stack([pers_all, off_all])),
]:
    m = np.isfinite(X).all(axis=1)
    tr, te = (train & m), (ok & m)
    model = HistGradientBoostingRegressor(
        loss="poisson", max_depth=4, max_iter=200, learning_rate=0.06, random_state=0)
    model.fit(X[tr], y_all[tr])
    pred = np.clip(model.predict(X[te]), 1e-6, None)
    demo.append({"spec": label, "poisson_dev": poisson_dev(y_all[te], pred),
                 "mean_pred": pred.mean(), "mean_actual": y_all[te].mean()})

print(pd.DataFrame(demo).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nA tree given ONLY persistence reproduces the persistence floor almost exactly.")
print("Adding log_area as a plain feature makes it over-predict by ~68% and degrades deviance.")

## Does prior burn predict ignitions? A descriptive read

The student's W6 hypothesis: a hex that burned recently has had its fuel consumed, so it should be
*less* likely to carry a subsequent fire. `src/burn_history.py` builds that state from **perimeters**
— a point says where a fire started, not which hexes it consumed, and two thirds of MTBS-mapped
fires cross more than one hex.

This is deliberately a **descriptive** comparison of observed means, not a model. It therefore does
not depend on the Poisson problem or on any ablation specification.

In [ ]:
# Burn history is already on the cached panel; `bp` is an alias kept so the
# cells below read as one continuous analysis.
bp = panel
bp_test = bp[bp["season_year"] >= cfg.test_start]

for f in ["any_burn_lag4", "any_burn_lag12", "any_burn_lag20"]:
    g = bp_test.groupby(f)["starts_natural"].agg(["mean", "size"])
    ratio = g["mean"].iloc[1] / g["mean"].iloc[0]
    yrs = int(f.split("lag")[1]) // 4
    print(f"{f}  ({yrs}-year window)")
    print(f"   unburned  mean {g['mean'].iloc[0]:.4f}  n {g['size'].iloc[0]:>9,}")
    print(f"   burned    mean {g['mean'].iloc[1]:.4f}  n {g['size'].iloc[1]:>9,}")
    print(f"   ratio     {ratio:.2f}x more natural ignitions in recently-burned hexes\n")

**The sign is the opposite of the hypothesis.** Recently-burned hexes carry roughly **2.9× more**
natural ignitions, not fewer.

The obvious objection is confounding: prior burn may simply mark fire-prone terrain, in which case
the ratio says nothing beyond "fires happen where fires happen." The cell below tests exactly that,
by stratifying on the persistence baseline — comparing burned against unburned hexes *among hexes
with similar ignition histories*.

In [ ]:
s = bp[(bp["season_year"] >= cfg.test_start) & bp["pers_natural"].notna()].copy()

# Rank-then-qcut: the raw persistence values are ~84% ties at zero, which qcut
# cannot split into equal bins directly.
s["band"] = pd.qcut(s["pers_natural"].rank(method="first"), 5,
                    labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])

t = s.groupby(["band", "any_burn_lag12"])["starts_natural"].agg(["mean", "size"]).unstack(1)
within = (t["mean"][1] / t["mean"][0]).rename("burned/unburned")

print("Mean natural starts, by persistence band x prior burn (3-year window)\n")
print(t.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nWithin-band ratio — the confound-controlled version:\n")
print(within.round(2).to_string())

**Finding — the association survives the control.** Within every persistence quintile, recently
burned hexes carry **1.4× to 2.4×** more natural ignitions. Prior burn is not merely a proxy for
baseline ignition propensity; it carries information persistence does not.

That leaves two competing readings of *what* prior burn measures, and they are not the same variable:

- **Dynamic fuel state** — the W6 hypothesis. A burn consumes fuel, so ignition should be suppressed
  immediately afterwards and recover as vegetation regrows.
- **Static terrain quality** — prior burn simply marks terrain that carries fire, and such terrain
  keeps carrying fire regardless of when it last burned.

The section below separates them, because they make **opposite predictions about timing**.

## Static terrain, or dynamic fuel? The decay test

`seasons_since_burn` distinguishes the two readings directly:

| reading | prediction for time since burn |
| --- | --- |
| fuel depletion | **strong suppression at year 0**, recovering over ~5–15 years as fuel regrows |
| terrain quality | **flat** — a hex that burned 20 years ago is still fire-carrying terrain |

This is a descriptive comparison of observed means, so it does not depend on the Poisson problem or
on any ablation specification.

**One methodological trap, found the hard way.** Run naively across all seasons, this analysis
produces a clean alternating high-low pattern that looks like a real biennial signal. It is an
**aliasing artifact**: `seasons_since_burn` counts *seasons*, most perimeter burns occur in JJA, and
JJA carries 0.44 natural starts per cell against DJF's 0.003. So consecutive values of
`seasons_since_burn` land on systematically different seasons, and the season effect masquerades as
a decay pattern. The cell below shows the alias, then the analysis restricts to **JJA only** to
remove it.

In [ ]:
alias = bp[(bp["season_year"] >= cfg.test_start) & (bp["seasons_since_burn"] < 116)]

print("Mean natural starts by season (0=DJF, 1=MAM, 2=JJA, 3=SON):")
print(alias.groupby("season_ord")["starts_natural"].mean().round(4).to_string())

print("\nSeason composition of each (seasons_since_burn % 4) value:")
print(pd.crosstab(alias["seasons_since_burn"] % 4, alias["season_ord"],
                  normalize="index").round(3).to_string())
print("\nEach offset is dominated by a different season — hence the spurious alternation.")

In [ ]:
# JJA only: the season alias cannot operate within a single season.
jja = bp[(bp["season_year"] >= cfg.test_start)
         & (bp["season_ord"] == 2)
         & bp["pers_natural"].notna()].copy()

NEVER_BURNED = 116  # the censor value used by src/burn_history.py
never = jja[jja["seasons_since_burn"] >= NEVER_BURNED]
ever = jja[jja["seasons_since_burn"] < NEVER_BURNED].copy()
base = never["starts_natural"].mean()

ever["yrs"] = (ever["seasons_since_burn"] / 4).round().astype(int)
decay = ever.groupby("yrs")["starts_natural"].agg(["mean", "size"])
decay = decay[decay["size"] >= 1500]          # drop bins too thin to read
decay["vs_never"] = decay["mean"] / base

print(f"JJA only. Never-burned reference: n={len(never):,}, mean={base:.4f}\n")
print("Natural starts by years since last perimeter burn:")
print(decay.to_string(float_format=lambda x: f"{x:.4f}"))

In [ ]:
# The same question with baseline propensity held fixed: read DOWN a column for the
# time effect, ACROSS a row for the propensity effect.
ever["band"] = pd.qcut(ever["pers_natural"].rank(method="first"), 4,
                       labels=["Q1", "Q2", "Q3", "Q4"])
ever["bucket"] = pd.cut(ever["yrs"], [-1, 1, 3, 6, 10, 30],
                        labels=["0-1", "2-3", "4-6", "7-10", "11+"])

grid = ever.pivot_table(index="bucket", columns="band", values="starts_natural",
                        aggfunc="mean", observed=True)
print("JJA mean natural starts: years since burn x persistence quartile\n")
print(grid.to_string(float_format=lambda x: f"{x:.4f}"))

spread_time = (grid.max(axis=0) / grid.min(axis=0)).mean()
spread_prop = (grid.max(axis=1) / grid.min(axis=1)).mean()
print(f"\nmean within-column spread (time effect):       {spread_time:.2f}x")
print(f"mean within-row spread (propensity effect):    {spread_prop:.2f}x")

**Finding — prior burn is static terrain quality, not dynamic fuel state.**

The curve is **flat**: 4.2× the never-burned rate immediately after a burn, still ~3.5× eighteen
years later. There is a slight decline over time, but no recovery curve — and decisively, **no
suppression at year 0**. Under the fuel-depletion hypothesis, years 0–1 would be the *minimum*. They
are the maximum.

Holding persistence fixed makes it starker. Reading down any column, time since burn moves the rate
barely at all; reading across any row, propensity quartile moves it by an order of magnitude. Which
hex you are looking at dominates; when it last burned is close to irrelevant.

**What this settles, and what it costs.** The W6 fuel-consumption hypothesis is **not supported at
res-5 hex-season grain**. Prior burn identifies terrain that carries fire, and such terrain keeps
carrying fire. That also explains why burn history fought the persistence baseline in the ablation:
both encode the same static property, and persistence encodes it *better* — it is built from all
2.27M ignitions, while burn history sees only the 0.6% of fires with MTBS perimeters.

**Consequences for the feature set:**

- `seasons_since_burn` is near-constant once propensity is controlled and should probably be dropped.
- `any_burn_lag*` may still earn a place as a **terrain marker** for hexes whose ignition history is
  too sparse for persistence to characterise — the within-band lift is real — but it is not the
  fuel-state covariate it was built to be.
- The **fuel-condition** question is untouched by this result. Prior burn measures fuel *history*;
  the climate layer measures how dry the fuel *is*. That hypothesis remains open and untested at
  this grain.

**Scope of the claim.** This is res-5 hexes (~62,494 acres), perimeter-linked burns, 2010+ held-out
JJA. A depletion effect operating at finer spatial grain, or over intervals shorter than one season,
would not be visible here.

## Does hex-grain climate beat persistence?

**This is the question the entire hex build was for.** W4 found that TerraClimate covariates added
nothing at EPA Level III grain, but the per-region stratification in
[`07_natural_location.ipynb`](07_natural_location.ipynb) suggested the grain was the problem rather
than the data: per-region Spearman ran 0.086–0.529 and *inverted sign* in two regions, so pooling
105 regions averaged a real signal to zero. The reading was "the covariates are real but the grain
of the model is wrong."

Hex grain tests that directly. ~15.8 TerraClimate cells average into each res-5 hex, so the
covariate is genuinely resolved at the unit being predicted.

**Scored on Spearman, not deviance.** The target is 85–96% zeros and 4.3–9.3× overdispersed, which
makes Poisson deviance unreliable here. Rank correlation is invariant to monotone transforms — a
tree given only persistence scores an identical 0.3435 under Poisson, squared-error, or a hurdle
classifier — and it matches the siting decision, which consumes an ordering rather than an expected
count.

**Two forms of the climate feature**, because they ask different questions:

- **raw** — is this hex dry? Largely *cross-sectional*: deserts are dry every year.
- **anomaly** — is this hex drier *than its own normal*? The *temporal* part, and the only part
  persistence cannot already encode. Hex means are computed on training years only; using the full
  record would let the held-out period define the normal it is measured against.

In [ ]:
ANOM = [f"{c}_anom" for c in hp.CLIMATE_COVS]

rungs = {
    "+ climate (raw)": list(hp.CLIMATE_COVS),
    "+ climate (anomaly)": ANOM,
    "+ burn history": list(hp.BURN_COVS),
    "+ climate + burn": list(hp.CLIMATE_COVS) + list(hp.BURN_COVS),
}

for target, season_ord, label in [
    ("starts_natural", 2, "NATURAL — JJA only (where natural fire lives)"),
    ("starts_natural", None, "NATURAL — all seasons"),
    ("starts_human", None, "HUMAN — all seasons"),
]:
    print("=" * 68)
    print(label)
    out = hp.ladder(panel, target, rungs, cfg=cfg, season_ord=season_ord)
    print(out.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
    print()

**Finding — no rung beats persistence, on any branch, in any season scope.**

Every covariate combination scores *below* the floor. The best case is burn history on JJA Natural
at −0.0038, which is a rounding error rather than lift; climate costs −0.038 to −0.110.

An ablation alone cannot distinguish "the model failed to use the covariate" from "the covariate
carries no usable signal." The cell below separates them by asking what is in the covariate before
any learner touches it.

In [ ]:
marg = hp.marginal_signal(
    panel, "starts_natural", list(hp.CLIMATE_COVS) + ANOM, cfg=cfg, season_ord=2
)
print("Model-free rank correlation with JJA natural starts (held-out years):\n")
print(marg.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

**Finding — the climate signal is real, physically coherent, and almost entirely
cross-sectional.**

The raw covariates carry the right physics: `pdsi` −0.137 (drier → more starts), `water_deficit`
+0.160, `vpd` +0.089. These are not noise.

But converting to a within-hex anomaly collapses them: `pdsi` −0.137 → **−0.073**, `water_deficit`
+0.160 → **−0.004**. Almost all of the apparent climate signal is the difference between *dry places
and wet places*, not between *dry years and wet years* in the same place.

**That is why the ladder fails.** Dry places are static geography, and a hex's own ignition history
already encodes static geography — more completely than four climate variables can, because it
integrates every driver at once. The covariates are largely re-describing what the baseline knows.

**This is a stronger null than W4's.** The W4 result could be dismissed as an artifact of averaging
over 105 heterogeneous ecoregions. This one tests that explanation directly, at a grain where the
covariate is properly resolved, and the covariates still do not add lift — with a model-free
measurement showing *why*.

**Scope.** What fails is *pre-season seasonal-mean dryness for predicting ignition counts*. Nothing
here speaks to whether the same covariates predict **burned area** given an ignition, which is a
different target and the one W4's megafire finding was about. Sub-seasonal timing — a dry spell in
the week before a lightning outbreak — is also invisible to a three-month seasonal mean.

## Fourth covariate: does pre-season vegetation density help?

The MODIS fuel-density probe, restricted to the six forest ecoregions it was fetched for. Same
ladder, same floor, same metric — so this rung is directly comparable to the climate rungs above.

Prior: the climate null was explained by its signal being cross-sectional rather than temporal.
NDVI has the same structure — forests are green every year — so the expectation is another null,
with the **anomaly** form as the only rung that could carry new information.

In [ ]:
ndvi = pd.read_parquet(DATA / "hex_season_ndvi.parquet")[["hex_id", "season_idx", "ndvi", "evi"]]
six = panel[panel["hex_id"].isin(set(ndvi["hex_id"]))].merge(
    ndvi, on=["hex_id", "season_idx"], how="left")

tr_years = six["season_year"] < cfg.test_start
norm = six[tr_years].groupby("hex_id")[["ndvi", "evi"]].mean()
for c in ["ndvi", "evi"]:
    six[f"{c}_anom"] = six[c] - six["hex_id"].map(norm[c])
six = six.sort_values(list(hp.HEX_SORT_KEYS)).reset_index(drop=True)

veg_rungs = {
    "+ NDVI (raw)": ["ndvi", "evi"],
    "+ NDVI (anomaly)": ["ndvi_anom", "evi_anom"],
    "+ climate": list(hp.CLIMATE_COVS),
    "+ NDVI + climate": ["ndvi", "evi", "ndvi_anom", "evi_anom"] + list(hp.CLIMATE_COVS),
}
print("IGNITION — JJA natural, six forest regions, held out 2010+\n")
print(hp.ladder(six, "starts_natural", veg_rungs, cfg=cfg, season_ord=2)
      .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
print("\nModel-free marginal signal:\n")
print(hp.marginal_signal(six, "starts_natural", ["ndvi", "evi", "ndvi_anom", "evi_anom"],
                         cfg=cfg, season_ord=2)
      .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

**Finding — a fourth covariate, the same null, and the same reason.**

Raw NDVI adds +0.004, the anomaly form *costs* −0.009, and the combination with climate is flat.
The marginals repeat the pattern the climate section established: raw NDVI correlates +0.228 with
JJA natural ignitions, but as a within-hex anomaly only +0.098. It measures which places grow
vegetation, not which years grow more of it — and which places grow vegetation is static geography
that a hex's own ignition history already encodes.

**Four independent covariates have now failed on this target** — drought, prior burn, vegetation
density, and their combinations. That is no longer four disappointments; it is a finding about the
phenomenon: **where lightning fires start, at this grain, is a property of the place rather than of
the year.**

Note this same covariate pair *does* carry information about **burned area** — see
[`13_hex_acres_baselines.ipynb`](13_hex_acres_baselines.ipynb). The distinction is the substantive
result: fuel state does not tell you where a fire starts, but it does say something about how much
burns once one does.

## What is NOT established here

**Resolved since the first draft.** An earlier version of this notebook excluded its ablation ladder
entirely, because the same rungs scored −30%, −115% and +3.4% across specifications. Three causes
were found:

1. **`log_area` passed as a feature rather than an offset** — demonstrated above; inflated
   predictions ~68%. Fixed.
2. **The persistence baseline re-typed inline on every run** — the harness was being silently
   re-specified between runs. Fixed by moving it to [`../src/hex_panel.py`](../src/hex_panel.py),
   which calls the W5 `trailing.TrailingMean`. The ladder above now reproduces to four decimals
   across runs.
3. **Poisson deviance against a 4.3–9.3× overdispersed, zero-inflated count.** Sidestepped rather
   than fixed: Spearman is invariant to monotone transforms, so the rank metric does not depend on
   getting the likelihood right.

**Still open.**

- **A dispersion-appropriate likelihood** (negative binomial or hurdle) if an *expected count* is
  ever needed rather than a ranking. The siting product needs only the ranking, so this is not on
  the critical path.
- **Burned area given ignition.** Everything here targets ignition *counts*. The W4 megafire finding
  — that persistence under-predicts record fire years by 1–1.7 orders of magnitude — is about
  *acres*, and this notebook does not speak to it. Pre-season dryness failing to predict *where
  fires start* is not evidence that it fails to predict *how big they get*.
- **Sub-seasonal climate timing.** A three-month pre-season mean cannot express a dry spell in the
  week before a lightning outbreak.

## Summary

1. **Ignition likelihood is predictable well above chance, and the skill is spatial.** Against
   shuffled persistence — same predicted values, wrong hexes — Spearman goes from −0.001 to
   **+0.53 (Human)** and **+0.34 (Natural)**.
2. **Human ranks better than Natural**, consistent with human ignition tracking near-static
   settlement geography while lightning depends on the season.
3. **Prior burn marks static terrain quality, not dynamic fuel state.** Recently-burned hexes carry
   ~2.9× more natural ignitions, the effect does not decay with time since burn (4.2× at year 0,
   ~3.5× at year 18), and holding propensity fixed the time effect is 1.30× against a propensity
   effect of 17.76×. The W6 fuel-depletion hypothesis is **not supported** at this grain.
4. **Hex-grain climate does not beat persistence.** No rung improves on the floor on any branch in
   any season scope. The model-free marginals explain why: the climate signal is physically coherent
   but almost entirely **cross-sectional** — `pdsi` −0.137 raw collapses to −0.073 as a within-hex
   anomaly, `water_deficit` +0.160 collapses to −0.004. It identifies dry *places*, not dry *years*,
   and dry places are static geography the baseline already encodes.

**What the project has, stated plainly.** A persistence-based ignition surface that ranks hexes far
better than chance, at a grain fine enough to site mitigation against — and a well-measured null on
whether pre-season climate improves it. Per the project's standing commitment, *a null is
publishable*; this one is now measured with an instrument that reproduces.

**What would change the answer.** A covariate carrying genuine *interannual* signal at hex grain —
which pre-season seasonal-mean dryness demonstrably does not. Fuel-condition imagery (NDVI/NBR) was
the original candidate and remains untested, though the result above lowers the prior: if seasonal
dryness is this cross-sectional, a vegetation index composited over the same window may be too.